# SchemaSmith DPO on Tiny Aya

Free T4 notebook for Colab or Kaggle. It continues from an SFT LoRA and runs AlignTune DPO (`create_rl_trainer`, `algorithm="dpo"`, `backend="trl"`).

AlignTune 0.1.32 loads `model_name` and then adds a new LoRA. This notebook merges the SFT adapter into a local weight directory first, and passes that directory as `model_name`. If the adapter or a GPU is missing, the training cell returns.

Preference rows are planted failures, not model outputs. Do not read them as scores.


## Install

In [ ]:
import subprocess
import sys
from pathlib import Path

if not (Path.cwd() / "pyproject.toml").exists():
    raise SystemExit("Start the notebook from the repository root.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-lexsi.txt"]
)
print("install finished")


## Hugging Face login

In [ ]:
import os

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
try:
    from google.colab import userdata
    token = token or userdata.get("HF_TOKEN")
except Exception:
    pass

if token:
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("HF login ok")
else:
    print("No HF_TOKEN. Gated weights will not download.")


## Preference data

Uses the train split when it exists. Otherwise builds a fresh 24-row set and keeps its train split.

In [ ]:
from pathlib import Path

from schemasmith.curation.local import curate_local
from schemasmith.data.generator import generate_tickets
from schemasmith.data.loader import load_alpaca_jsonl
from schemasmith.data.preferences import write_dpo_dataset

train_file = Path("data/curated/local/train/sft_alpaca.jsonl")
if train_file.is_file():
    train_rows = load_alpaca_jsonl(train_file)
else:
    curated = curate_local(generate_tickets(24, seed=42), Path("data/notebook-dpo/local"), seed=42)
    train_rows = curated.splits["train"]

summary = write_dpo_dataset(train_rows, Path("data/dpo"), seed=42)
print(summary["pair_count"], summary["split_counts"], summary["failure_modes"])


## Merge the SFT adapter

Set `SFT_ADAPTER` to the adapter directory from the SFT notebook. The merge runs only on a GPU.

In [ ]:
import os
from pathlib import Path

import torch

SFT_ADAPTER = os.environ.get("SFT_ADAPTER", "output/lora-tiny-aya")
MERGED = Path("output/sft-merged")

if not torch.cuda.is_available():
    print("No GPU. Merge skipped.")
elif not Path(SFT_ADAPTER).exists():
    print(f"No SFT adapter at {SFT_ADAPTER}. Merge skipped.")
else:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from schemasmith.training import MODEL_ID

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )
    merged = PeftModel.from_pretrained(base, SFT_ADAPTER).merge_and_unload()
    MERGED.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(MERGED)
    tokenizer.save_pretrained(MERGED)
    print("merged", MERGED)


## Train DPO

In [ ]:
from pathlib import Path

import torch
from aligntune.core.backend_factory import create_rl_trainer

from schemasmith.training import lora_dpo_kwargs

dataset_dir = Path("data/dpo")
merged = Path("output/sft-merged")
if not torch.cuda.is_available():
    print("No GPU. DPO skipped.")
elif not merged.is_dir():
    print("Merged SFT weights are missing. DPO skipped.")
elif not (dataset_dir / "train" / "dpo.jsonl").is_file():
    print("Preference data is missing. DPO skipped.")
else:
    kwargs = lora_dpo_kwargs(
        dataset_dir=str(dataset_dir.resolve()),
        output_dir="output/lora-tiny-aya-dpo",
        model_name=str(merged.resolve()),
    )
    print(kwargs["algorithm"], kwargs["backend"], kwargs["config_name"], kwargs["use_peft"])
    trainer = create_rl_trainer(**kwargs)
    trainer.train()
    print("train() returned")


## Not a score

This notebook does not measure the base model or the adapters. Run `schemasmith compare` after both adapters exist. Until `reports/compare.json` exists, leave the README table as TBD.
